# Recommender Systems

In [78]:
import pandas as pd
import numpy as np

df = pd.read_csv('movies.csv')
print('Shape:', df.shape)
df.head()

Shape: (103, 10)


,id,title,genres,overview,vote_count,vote_average,popularity,runtime,budget,revenue
0,1,The Dark Knight,Action Crime Drama Thriller,Batman raises the stakes in his war on crime w...,12269,8.5,123.2,192,204900595,1945803228
1,2,The Shawshank Redemption,Crime Drama,Two imprisoned men bond over a number of years...,8358,8.5,51.6,104,132521863,433734972
2,3,The Godfather,Crime Drama,Spanning ten years the aging patriarch of an o...,6024,8.4,41.1,110,138224038,1924837113
3,4,Pulp Fiction,Crime Drama Thriller,The lives of two mob hitmen a boxer a gangster...,8670,8.3,140.9,172,254467210,1982458954
4,5,Schindler's List,Drama History War,In German-occupied Poland during World War II ...,4436,8.3,40.3,177,81737383,903664919


Simple Recommender

In [79]:
C = df['vote_average'].mean()
print(f'Global mean rating C = {C:.2f}')

Global mean rating C = 7.79


In [80]:
m = df['vote_count'].quantile(0.90)
print(f'Minimum votes required m = {m:.0f}')

Minimum votes required m = 12873


In [81]:
qualified = df[df['vote_count'] >= m].copy()
print(f'Movies that qualify: {len(qualified)} out of {len(df)}')

Movies that qualify: 11 out of 103


In [ ]:
def weighted_rating(row, C=C, m=m):
    
    v = row['vote_count']
    R = row['vote_average']
    return (v / (v + m)) * R + (m / (v + m)) * C # weighted rating (WR)=(v/(v+m))R+(m/(v+m))C used on IMDB

qualified['score'] = qualified.apply(weighted_rating, axis=1)
qualified.head(3)[['title', 'vote_count', 'vote_average', 'score']]

,title,vote_count,vote_average,score
6,Inception,14075,8.1,7.952053
18,Avengers: Endgame,15901,8.3,8.071965
19,Avengers: Infinity War,16376,8.3,8.075668


In [83]:
top_movies = (
    qualified
    .sort_values('score', ascending=False)
    [['title', 'vote_count', 'vote_average', 'score']]
    .head(15)
    .reset_index(drop=True)
)

top_movies.index += 1
top_movies.index.name = 'rank'
top_movies.round(3)

,title,vote_count,vote_average,score
rank,,,,
1,Avengers: Infinity War,16376,8.3,8.076
2,Avengers: Endgame,15901,8.3,8.072
3,Joker,19668,8.2,8.038
4,Joker,19668,8.2,8.038
5,Inception,14075,8.1,7.952
6,Guardians of the Galaxy,13786,8.0,7.899
7,Thor: Ragnarok,13168,7.7,7.745
8,Spider-Man: Homecoming,13024,7.4,7.594
9,Wonder Woman,13044,7.4,7.594


In [84]:
def build_chart(genre, percentile=0.85):

    genre_df = df[df['genres'].str.contains(genre, case=False, na=False)].copy()
    
    m_genre = genre_df['vote_count'].quantile(percentile)
    C_genre = genre_df['vote_average'].mean()
    
    qualified_genre = genre_df[genre_df['vote_count'] >= m_genre].copy()
    
    qualified_genre['score'] = qualified_genre.apply(
        lambda row: weighted_rating(row, C=C_genre, m=m_genre), axis=1
    )
    
    return (
        qualified_genre
        .sort_values('score', ascending=False)
        [['title', 'genres', 'vote_count', 'vote_average', 'score']]
        .head(10)
        .reset_index(drop=True)
    )

print('=== Top Action movies ===')
build_chart('Action').round(3)

=== Top Action movies ===


,title,genres,vote_count,vote_average,score
0,Avengers: Infinity War,Action Adventure Science Fiction,16376,8.3,8.006
1,Avengers: Endgame,Action Adventure Science Fiction,15901,8.3,8.001
2,Inception,Action Adventure Science Fiction Thriller,14075,8.1,7.879
3,Guardians of the Galaxy,Action Adventure Comedy Science Fiction,13786,8.0,7.826
4,Black Panther,Action Adventure Science Fiction Fantasy,14387,7.3,7.468


In [85]:
print('=== Top Drama movies ===')
build_chart('Drama').round(3)

=== Top Drama movies ===


,title,genres,vote_count,vote_average,score
0,The Dark Knight,Action Crime Drama Thriller,12269,8.5,8.221
1,Parasite,Comedy Drama Thriller,11802,8.5,8.215
2,Joker,Crime Drama Thriller,19668,8.2,8.093
3,Joker,Crime Drama Thriller,19668,8.2,8.093
4,Inside Out,Animation Adventure Comedy Drama Family Fantasy,12261,8.1,8.002
5,Interstellar,Adventure Drama Science Fiction,11187,8.1,7.997
6,La La Land,Comedy Drama Music Romance,11085,7.9,7.892
7,Dune,Science Fiction Adventure Drama,10204,7.9,7.892
8,Arrival,Drama Mystery Science Fiction Thriller,10140,7.9,7.892
9,The Revenant,Adventure Drama Thriller Western,10148,7.4,7.641


In [86]:
print('=== Top Animation movies ===')
build_chart('Animation').round(3)

=== Top Animation movies ===


,title,genres,vote_count,vote_average,score
0,Inside Out,Animation Adventure Comedy Drama Family Fantasy,12261,8.1,7.973
1,Frozen,Animation Adventure Comedy Family Fantasy,12264,7.5,7.666


Content-Based Recommender

In [87]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=1,
)

df['overview'] = df['overview'].fillna('')

tfidf_matrix = tfidf.fit_transform(df['overview'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} movies × {tfidf_matrix.shape[1]} unique terms')

TF-IDF matrix shape: (103, 2160)
  → 103 movies × 2160 unique terms


In [88]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f'Cosine similarity matrix shape: {cosine_sim.shape}')
print('Sample (first 4×4 block):')
print(np.round(cosine_sim[:4, :4], 3))

Cosine similarity matrix shape: (103, 103)
Sample (first 4×4 block):
[[1.    0.    0.031 0.   ]
 [0.    1.    0.029 0.042]
 [0.031 0.029 1.    0.   ]
 [0.    0.042 0.    1.   ]]


In [89]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

def get_recommendations(title, cosine_sim=cosine_sim, top_n=10):

    if title not in indices:
        raise ValueError(f"'{title}' not found in dataset. Check spelling.")
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1 : top_n + 1]
    
    movie_indices = [s[0] for s in sim_scores]
    movie_scores  = [s[1] for s in sim_scores]
    
    result = df.iloc[movie_indices][['title', 'genres', 'vote_average']].copy()
    result.insert(0, 'similarity', np.round(movie_scores, 4))
    result = result.reset_index(drop=True)
    result.index += 1
    return result

In [90]:
print('=== Recommendations for: Inception ===')
get_recommendations('Inception')

=== Recommendations for: Inception ===


,similarity,title,genres,vote_average
1,0.0439,Minari,Drama,7.5
2,0.0324,The Suicide Squad,Action Adventure Comedy Fantasy Science Fiction,7.8
3,0.0319,Hereditary,Drama Horror Mystery Thriller,7.3
4,0.0299,Ford v Ferrari,Action Biography Drama Sport,8.1
5,0.0299,The Irishman,Biography Crime Drama,7.8
6,0.0000,The Dark Knight,Action Crime Drama Thriller,8.5
7,0.0000,The Shawshank Redemption,Crime Drama,8.5
8,0.0000,The Godfather,Crime Drama,8.4
9,0.0000,Pulp Fiction,Crime Drama Thriller,8.3
10,0.0000,Schindler's List,Drama History War,8.3


In [91]:
print('=== Recommendations for: The Dark Knight ===')
get_recommendations('The Dark Knight')

=== Recommendations for: The Dark Knight ===


,similarity,title,genres,vote_average
1,0.1005,The Dark Knight Rises,Action Crime Drama Thriller,7.6
2,0.0650,Joker,Crime Drama Thriller,8.2
3,0.0368,Joker,Crime Drama Thriller,8.2
4,0.0363,Uncut Gems,Crime Drama Thriller,7.5
5,0.0346,The Trial of the Chicago 7,Drama History Thriller,7.8
6,0.0331,Dune,Science Fiction Adventure Drama,7.9
7,0.0329,Baby Driver,Action Crime Music,7.6
8,0.0312,The Godfather,Crime Drama,8.4
9,0.0292,Schindler's List,Biography Drama History War,8.3
10,0.0268,Knives Out,Crime Drama Mystery,7.9


In [92]:
print('=== Recommendations for: Parasite ===')
get_recommendations('Parasite')

=== Recommendations for: Parasite ===


,similarity,title,genres,vote_average
1,0.0554,Hereditary,Drama Horror Mystery Thriller,7.3
2,0.0334,Minari,Drama,7.5
3,0.0327,Roma,Drama,7.7
4,0.0303,Gladiator,Action Adventure Drama,7.8
5,0.0277,Hell or High Water,Crime Drama,7.6
6,0.0265,Dune,Science Fiction Adventure Drama,7.9
7,0.0000,The Dark Knight,Action Crime Drama Thriller,8.5
8,0.0000,The Shawshank Redemption,Crime Drama,8.5
9,0.0000,The Godfather,Crime Drama,8.4
10,0.0000,Pulp Fiction,Crime Drama Thriller,8.3


In [93]:
print('=== Recommendations for: Interstellar ===')
get_recommendations('Interstellar')

=== Recommendations for: Interstellar ===


,similarity,title,genres,vote_average
1,0.0783,Dune,Science Fiction Adventure Drama,7.9
2,0.0480,Mad Max: Fury Road,Action Adventure Science Fiction Thriller,7.4
3,0.0472,The Avengers,Action Adventure Science Fiction,7.7
4,0.0459,The Revenant,Adventure Drama Thriller Western,7.4
5,0.0424,Room,Drama Thriller,8.1
6,0.0422,Mission Impossible - Fallout,Action Adventure Thriller,7.5
7,0.0419,The Departed,Crime Drama Thriller,7.9
8,0.0389,Avengers: Infinity War,Action Adventure Science Fiction,8.3
9,0.0385,Tenet,Action Science Fiction Thriller,7.4
10,0.0384,Frozen II,Animation Adventure Comedy Family Fantasy,6.9


Enhanced content-based: combine overview + genres

In [94]:
df['soup'] = df['overview'] + ' ' + df['genres'].fillna('')

tfidf_enhanced = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix_enhanced = tfidf_enhanced.fit_transform(df['soup'])
cosine_sim_enhanced = cosine_similarity(tfidf_matrix_enhanced, tfidf_matrix_enhanced)

print(f'Enhanced TF-IDF matrix shape: {tfidf_matrix_enhanced.shape}')

Enhanced TF-IDF matrix shape: (103, 2331)


In [95]:
print('=== Enhanced recommendations for: Inception ===')
get_recommendations('Inception', cosine_sim=cosine_sim_enhanced)

=== Enhanced recommendations for: Inception ===


,similarity,title,genres,vote_average
1,0.0883,Mad Max: Fury Road,Action Adventure Science Fiction Thriller,7.4
2,0.0676,The Avengers,Action Adventure Science Fiction,7.7
3,0.0672,Spider-Man: Homecoming,Action Adventure Science Fiction,7.4
4,0.0658,Jurassic World,Action Adventure Science Fiction Thriller,6.7
5,0.0651,The Suicide Squad,Action Adventure Comedy Fantasy Science Fiction,7.8
6,0.0645,Spider-Man: Into the Spider-Verse,Animation Action Adventure Science Fiction,8.4
7,0.0632,Captain America: Civil War,Action Adventure Science Fiction,7.5
8,0.0620,Avengers: Endgame,Action Adventure Science Fiction,8.3
9,0.0586,Star Wars: The Last Jedi,Action Adventure Science Fiction Fantasy,7.0
10,0.0581,Iron Man,Action Adventure Science Fiction,7.4


In [96]:
print('=== Enhanced recommendations for: The Dark Knight ===')
get_recommendations('The Dark Knight', cosine_sim=cosine_sim_enhanced)

=== Enhanced recommendations for: The Dark Knight ===


,similarity,title,genres,vote_average
1,0.1784,The Dark Knight Rises,Action Crime Drama Thriller,7.6
2,0.1175,Joker,Crime Drama Thriller,8.2
3,0.1024,Joker,Crime Drama Thriller,8.2
4,0.0884,Uncut Gems,Crime Drama Thriller,7.5
5,0.0808,Baby Driver,Action Crime Music,7.6
6,0.0729,Promising Young Woman,Crime Drama Thriller,7.5
7,0.0721,John Wick: Chapter 3,Action Crime Thriller,7.5
8,0.0685,Pulp Fiction,Crime Drama Thriller,8.3
9,0.0654,The Departed,Crime Drama Thriller,7.9
10,0.0602,The Godfather,Crime Drama,8.4


In [97]:
print('=== Enhanced recommendations for: Toy Story 3 ===')
get_recommendations('Toy Story 3', cosine_sim=cosine_sim_enhanced)

=== Enhanced recommendations for: Toy Story 3 ===


,similarity,title,genres,vote_average
1,0.1387,Frozen,Animation Adventure Comedy Family Fantasy,7.5
2,0.1383,Toy Story 4,Animation Adventure Comedy Family Fantasy,7.8
3,0.1203,Coco,Animation Adventure Comedy Family Fantasy Music,8.4
4,0.1197,Luca,Animation Adventure Comedy Family Fantasy,7.5
5,0.1190,Frozen II,Animation Adventure Comedy Family Fantasy,6.9
6,0.1068,Raya and the Last Dragon,Animation Action Adventure Comedy Family Fantasy,7.4
7,0.0800,Inside Out,Animation Adventure Comedy Drama Family Fantasy,8.1
8,0.0763,Soul,Animation Adventure Comedy Drama Fantasy Music,8.1
9,0.0354,Thor: Ragnarok,Action Adventure Comedy Science Fiction,7.7
10,0.0335,The Suicide Squad,Action Adventure Comedy Fantasy Science Fiction,7.8
